# Facebook Engagement Analyses
## Load Libraries

In [1]:
from fbri.private.sql.query import execute
import pandas as pd
import numpy as np
from scipy import stats
import os

## Define Functions

In [2]:
def ols_from_cells(cells, metric, design):

    k = len(next(iter(design.values())))
    X_rows = np.vstack([design[g] for g in cells["_group"]])
    n = cells["n"].to_numpy(float)
    s = cells[f"s_{metric}"].to_numpy(float)
    ss = cells[f"ss_{metric}"].to_numpy(float)
 
    XtX = (X_rows * n[:, None]).T @ X_rows
    Xty = (X_rows * s[:, None]).sum(axis=0)
    beta = np.linalg.solve(XtX, Xty)
 
    fitted = X_rows @ beta                      # per-cell fitted value
    # cluster (domain) score vectors: sum over cells of x * (s - n * fitted)
    cell_resid_sum = s - n * fitted             # sum of residuals within cell
    scores = X_rows * cell_resid_sum[:, None]
    score_by_domain = (
        pd.DataFrame(scores, columns=range(k))
        .assign(domain=cells["domain"].to_numpy())
        .groupby("domain").sum().to_numpy()
    )
    meat = score_by_domain.T @ score_by_domain
 
    N = n.sum()
    G = cells["domain"].nunique()
    XtX_inv = np.linalg.inv(XtX)
    cr1 = (G / (G - 1)) * ((N - 1) / (N - k))
    V = cr1 * XtX_inv @ meat @ XtX_inv
    se = np.sqrt(np.diag(V))
 
    tval = beta / se
    df = G - 1
    p = 2 * stats.t.sf(np.abs(tval), df)
    ci = stats.t.ppf(0.975, df) * se
    return pd.DataFrame({
        "metric": metric, "beta": beta, "se_cluster": se,
        "t": tval, "p": p,
        "conf_low": beta - ci, "conf_high": beta + ci,
        "N": int(N), "G_domains": int(G),
    })

## Initialize Global Variables

In [3]:
database_meta = "fbri_prod_private"
breakdowns_table = "ERC_CONDOR_URL_BREAKDOWNS_DP_CLEAN_PARTITIONED_V2"
 
database_user = "private_user_10160008500692067"        
analysis_table = "url_shares_test_analysis_variables" 

In [4]:
# breakdown metrics
METRICS = ["shares", "angers", "sorrys", "hahas", "wows", "loves"]

## Query the Breakdown Table

In [5]:
metric_sums = ",\n            ".join(
    f"SUM({m}) AS {m}" for m in METRICS)
agg_cols = ",\n        ".join(
    f"SUM(l.{m}) AS s_{m}, SUM(l.{m} * l.{m}) AS ss_{m}" for m in METRICS)
 
sql = f"""
    WITH link AS (
        SELECT url_rid,
            {metric_sums}
        FROM {database_meta}.{breakdowns_table}
        WHERE c = 'US'
        GROUP BY url_rid
    )
    SELECT
        t.domain,
        t.moral_agent,
        COUNT(*) AS n,
        {agg_cols}
    FROM {database_user}.{analysis_table} AS t
    JOIN link AS l ON t.url_rid = l.url_rid
    GROUP BY t.domain, t.moral_agent
"""

In [6]:
# save the results
cells = execute(sql)
print(f"Aggregate cells: {len(cells):,} rows "
      f"({cells['domain'].nunique():,} domains)")
cells.to_csv("facebook_engagement_analysis_data.csv", index=False)

[NOTICE] 1 output(s) filtered out

## Analyses

In [7]:
AGENT_DESIGN = {
    "neither":      np.array([1, 0, 0, 0]),
    "dyad":         np.array([1, 1, 0, 0]),
    "villain_only": np.array([1, 0, 1, 0]),
    "victim_only":  np.array([1, 0, 0, 1]),
}
AGENT_TERMS = ["intercept (neither)", "dyad", "villain_only", "victim_only"]
 
agent_cells = cells.copy()
agent_cells["_group"] = agent_cells["moral_agent"]
 
agent_results = []
for m in METRICS:
    res = ols_from_cells(agent_cells, m, AGENT_DESIGN)
    res["term"] = AGENT_TERMS
    agent_results.append(res)
agent_results = pd.concat(agent_results, ignore_index=True)
 
print(agent_results[agent_results["term"] != "intercept (neither)"]
      .round(4).to_string(index=False))
agent_results.to_csv("facebook_engagement_analysis_data.csv", index=False)

[NOTICE] 1 output(s) filtered out

In [8]:
# post-process: betas in outcome-SD units
def add_std_beta(results, cells):
    out = []
    for m in results["metric"].unique():
        N = cells["n"].sum()
        mu = cells[f"s_{m}"].sum() / N
        sd = np.sqrt(cells[f"ss_{m}"].sum() / N - mu**2)
        r = results[results["metric"] == m].copy()
        for col in ["beta", "se_cluster", "conf_low", "conf_high"]:
            r[col + "_std"] = r[col] / sd
        out.append(r)
    return pd.concat(out, ignore_index=True)

In [9]:
results_std = add_std_beta(agent_results, cells)

In [10]:
results_std

[NOTICE] 1 output(s) filtered out